# Tracking Habitat and Land Change from Space

Satellites photograph the same ground over and over. If we can teach a model to read those tiles and label what is on the ground, we get a running record of how a landscape changes over time — where forest gives way to cropland, where water shrinks or spreads, where urban areas push outward. That record is the raw material for conservation work: it tells you *where* habitat is being lost and *how fast*.

This notebook tells one continuous story: we train a small image classifier to sort satellite tiles into four **land cover / land use** classes and then watch it work.

- **forest** — tree-covered land
- **water** — lakes, rivers, sea
- **cropland** — cultivated fields
- **urban** — built-up, developed land

We use a small subset of the public **EuroSAT** dataset (Sentinel-2 tiles), a compact PyTorch CNN trained from scratch, and an Amazon SageMaker **managed training job**. The goal is a clear, working, end-to-end run — not state-of-the-art accuracy.

## Where Kiro helps

Kiro works alongside you right here inside SageMaker JupyterLab. Along the way it can:

- **Generate the training script** (`train.py`) — the CNN, the training loop, and the SageMaker input/output wiring.
- **Wire up the estimator** — filling in entry point, instance type, framework version, and hyperparameters correctly.
- **Explain SageMaker concepts** — what the execution role is for, why data goes to S3, and what a managed training job actually does.

The notebook runs top to bottom. Each code cell is introduced by a short note on *why* the step exists before the *how*.

## 1. Set up the SageMaker session

Before we can launch anything, SageMaker needs two things, and it is worth being explicit about why:

- **An execution role (IAM role).** A managed training job runs on infrastructure that AWS spins up on your behalf — separate from this notebook. That infrastructure needs permission to read your training data from S3 and to write the model artifact back. The **execution role** carries those permissions. Inside SageMaker JupyterLab we can resolve the role the environment already provides, so there is nothing to create by hand.
- **An S3 bucket.** The training job cannot read files from this notebook's local disk; it reads from Amazon S3. We use SageMaker's **default bucket** to stage the training data and to receive the model artifact, so no bucket setup is required.

The cell below creates the session, resolves the role, and picks the default bucket, then prints both so you can see exactly what will be used.

In [ ]:
import sagemaker

# A Session ties together the AWS clients SageMaker needs (S3, SageMaker, etc.).
session = sagemaker.Session()

# Resolve the execution role the JupyterLab environment already provides.
# This role is what the managed training job will assume to reach S3.
role = sagemaker.get_execution_role()

# Use the account/region default bucket to stage data and receive artifacts.
bucket = session.default_bucket()

print("Execution role:", role)
print("Default bucket: ", bucket)

## 2. Prepare a small EuroSAT subset

EuroSAT ships ten land-use categories; we only need four. `data_prep` downloads EuroSAT once (torchvision caches it), maps four source categories onto our target classes (Forest→forest, SeaLake→water, AnnualCrop→cropland, Industrial→urban), and caps the number of images per class so the download, upload, and training all stay quick.

It also sets aside a few **held-out** tiles per class *in memory*. Those tiles are never written for training and never uploaded — we keep them for the "does it actually work?" check after training, on tiles the model has not seen.

When prep finishes we print the per-class counts so you can see the subset composition.

In [ ]:
import data_prep

# Backed by torchvision; the download happens lazily on first use.
image_source = data_prep.load_eurosat_source()

# Build the ImageFolder-style subset under ./data and keep the held-out tiles.
data_dir = "data"
counts, held_out = data_prep.prepare_subset(image_source, out_dir=data_dir)

print("Images prepared per class:")
for class_name, n in counts.items():
    print(f"  {class_name:>8}: {n}")
print("Total images:", sum(counts.values()))
print("Held-out tiles (kept in memory, not uploaded):", len(held_out))

## 3. Preview a few sample tiles

Before training, it helps to *see* what the model learns from. The grid below shows a handful of tiles per class, each titled with its label. This runs purely in the notebook on the images we just wrote locally — it does not touch the training job at all.

In [ ]:
import os
from PIL import Image
from viz import show_sample_grid
from data_prep import TARGET_CLASSES

# Grab the first couple of tiles from each class directory for the preview.
samples = []
for class_name in TARGET_CLASSES:
    class_dir = os.path.join(data_dir, class_name)
    file_names = sorted(os.listdir(class_dir))[:2]
    for file_name in file_names:
        img = Image.open(os.path.join(class_dir, file_name)).convert("RGB")
        samples.append((img, class_name))

show_sample_grid(samples)

## 4. Upload the subset to S3

The managed training job reads its data from S3, not from this notebook's disk, so we upload the prepared subset to the default bucket under a demo-scoped prefix. The returned S3 URI becomes the training job's **input channel**.

Note the held-out tiles stay in memory — we deliberately do **not** upload them, so the post-training check runs on tiles the model never trained on.

In [ ]:
# Upload ./data to s3://<bucket>/satellite-land-cover-demo/data and capture the URI.
s3_data_uri = session.upload_data(
    path=data_dir,
    bucket=bucket,
    key_prefix="satellite-land-cover-demo/data",
)

print("Training data uploaded to:", s3_data_uri)

## 5. Configure the PyTorch estimator

The estimator describes the job we want SageMaker to run. The key arguments:

- **`entry_point="train.py"`** and **`source_dir="."`** — the script to run and the folder to ship to the training container (so `train.py` and its neighbors go along).
- **`role`** — the execution role from step 1; the job assumes it to reach S3.
- **`instance_type="ml.m5.large"`, `instance_count=1`** — a small, single CPU instance. Our model and subset are tiny, so CPU is plenty.
- **`framework_version="2.2"`, `py_version="py310"`** — pin the managed PyTorch container image, so `train.py` runs against a known PyTorch.
- **`hyperparameters`** — passed to `train.py` as command-line args; kept small (3 epochs, batch size 32) to keep the job short.
- **`output_path`** — where SageMaker writes the resulting `model.tar.gz`.

In [ ]:
from sagemaker.pytorch import PyTorch

estimator = PyTorch(
    entry_point="train.py",
    source_dir=".",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    framework_version="2.2",
    py_version="py310",
    hyperparameters={
        "epochs": 3,
        "batch-size": 32,
    },
    output_path=f"s3://{bucket}/",
)

print("Estimator configured with entry point:", estimator.entry_point)

## 6. Launch the managed training job

Calling `fit()` hands everything to SageMaker: it provisions the instance, pulls the container, ships our code, streams the S3 input channel in, runs `train.py`, and collects the model artifact. Training runs on that managed infrastructure — **not** inside this notebook process.

On success we print `estimator.model_data`, the S3 location of the trained `model.tar.gz`. If the job fails, `fit()` raises; we catch it and read the SageMaker **FailureReason** straight from the training job description so you see *why* it failed rather than a bare traceback.

In [ ]:
import boto3
from reporting import extract_failure_reason

try:
    estimator.fit({"training": s3_data_uri})
    print("Training complete.")
    print("Model artifact:", estimator.model_data)
except Exception as error:
    print("Training job raised an exception:", error)
    # Read the training job description from SageMaker (this part needs AWS),
    # then extract the failure reason with the pure, testable helper.
    sm_client = boto3.client("sagemaker")
    job_name = estimator.latest_training_job.name
    description = sm_client.describe_training_job(TrainingJobName=job_name)
    print("SageMaker FailureReason:", extract_failure_reason(description))

## 7. Load the trained model back into the notebook

To *see* the model work, we bring its artifact back locally. We download `model.tar.gz` from `estimator.model_data`, extract `model.pth`, rebuild the same `SmallCNN` architecture, and load the saved weights. This is purely for visualization — it reads the finished artifact and does not change the training job in any way.

The S3 download is injected as a small `download_fn` so the reusable helper in `viz.py` stays free of any hard SageMaker dependency.

In [ ]:
from sagemaker.s3 import S3Downloader
from viz import load_trained_model


def download_fn(model_data_uri, local_tar_path):
    """Fetch model.tar.gz from S3 into a local path (injected into viz)."""
    # S3Downloader.download writes into a directory; point it at the CWD and
    # rely on the artifact already being named model.tar.gz.
    S3Downloader.download(model_data_uri, ".")


model = load_trained_model(estimator.model_data, download_fn)
print("Model loaded and set to eval mode:", type(model).__name__)

## 8. Predicted vs. actual on held-out tiles

Finally, the payoff. We run the loaded model on the held-out tiles we set aside during prep — tiles it never trained on — and draw a grid annotating each with its **predicted** and **actual** label. This is the visible "it works" check, and like the earlier preview it runs entirely on local images without touching the training job.

In [ ]:
from torchvision import transforms
from viz import show_prediction_grid

# EuroSAT tiles are 64x64 RGB; Resize keeps any off-size tile at the shape the
# model expects, and ToTensor produces the (3, 64, 64) tensor.
to_tensor = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

show_prediction_grid(model, held_out, to_tensor)